# Sudoku-1 : Resolution par Backtracking

**Navigation** : [<< Sudoku-0 Environment](Sudoku-0-Environment.ipynb) | [Index](README.md) | [Sudoku-2 Genetic >>](Sudoku-2-Genetic.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Implementer** un algorithme de backtracking pour resoudre le Sudoku
2. **Comprendre** l'exploration en profondeur et le retour arriere
3. **Analyser** les performances du backtracking selon la difficulte des puzzles

**Duree estimee** : ~7 min | **Prerequis** : [Sudoku-0 Environment](Sudoku-0-Environment.ipynb) | **Lien** : Voir [Search-6 CSP Fundamentals](../Search/Foundations/Search-6-CSP-Fundamentals.ipynb) pour la theorie du backtracking

---

## Introduction Theorique

L'algorithme de backtracking est une methode de recherche en profondeur utilisee pour resoudre les problemes de satisfaction de contraintes (CSP), comme le Sudoku. L'algorithme explore toutes les configurations possibles pour trouver une solution qui respecte les contraintes :

- **Exploration en profondeur** : L'algorithme explore chaque possibilite de maniere exhaustive avant de revenir en arriere (backtrack) lorsque aucune solution n'est trouvee dans une branche particuliere.
- **Contraintes** : Dans le cas du Sudoku, les contraintes sont les regles du jeu : chaque chiffre de 1 a 9 doit apparaitre une seule fois par ligne, colonne et sous-grille de 3x3.

## Implementation de l'Algorithme de Backtracking

L'algorithme suit ces etapes :
1. Trouver une case vide dans la grille.
2. Tenter de placer un chiffre (1-9) dans la case vide.
3. Verifier si ce chiffre respecte les contraintes.
4. Si oui, passer a la case suivante et repeter le processus.
5. Si non, essayer le chiffre suivant.
6. Si aucun chiffre ne convient, revenir en arriere (backtrack).

## Importation des Classes de Base


In [1]:
#!import Sudoku-0-Environment-Csharp.ipynb


The below script needs to be able to find the current output cell; this is an easy method to get it.

# Sudoku-0 : Environnement et Classes de Base (C#)

**Navigation** : [Index](README.md) | [Sudoku-1 Backtracking C# >>](Sudoku-1-Backtracking-Csharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre la structure de donnees `SudokuGrid` et ses methodes principales
2. Utiliser `ISudokuSolver` pour implementer un solveur de Sudoku
3. Exploiter `SudokuHelper` pour charger des grilles et tester des solveurs
4. Comparer les performances de plusieurs solveurs sur differentes difficultes

**Prerequis** : Notions de base en C# (.NET Interactive)  
**Duree estimee** : ~15 min

Installed Packages Plotly.NET, 5.1.0

## Définition de la classe SudokuGrid

Nous définissons ici la classe SudokuGrid qui représente une grille de Sudoku et fournit des méthodes pour manipuler et afficher les grilles.


### Interpretation : Structure de donnees pour la grille Sudoku

**Sortie obtenue** : La classe `SudokuGrid` encapsule toutes les operations de manipulation, validation et affichage d'une grille de Sudoku 9x9.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `Cells[9,9]` | int[,] | Stockage interne des valeurs (0 = vide) |
| `AllNeighbours` | 27 x 9 positions | Pre-calcul des voisins ligne/colonne/bloc |
| `CellNeighbours[9][9]` | ~20 positions chacune | Voisins directs de chaque cellule |
| `GetAvailableNumbers()` | int[] | Candidats valides pour une cellule |
| `NbErrors()` | int | Nombre de conflits + modifications erronees |

**Points cles** :
1. **Pre-calcul des voisins** : `AllNeighbours` et `CellNeighbours` sont calcules une seule fois a l'initialisation, evitant les recalculs couteux
2. **Conversion flexible** : Methodes pour convertir entre tableaux 1D, 2D et jagged arrays (utile pour differents formats de fichiers)
3. **Validation robuste** : `NbErrors` compte a la fois les doublons (ligne/colonne/bloc) et les modifications de indices pre-remplis
4. **Parsing tolerant** : `ReadMultiSudoku` accepte plusieurs formats (`.`, `X`, `-`, espaces)

> **Note technique** : La structure `CellNeighbours[i][j]` contient environ 20 positions (8 ligne + 8 colonne + 4 bloc, moins les doublons). Ce pre-calcul est crucial pour les performances des algorithmes de backtracking et de propagation de contraintes.

...

### Interpretation : Interface de strategie

**Sortie obtenue** : L'interface `ISudokuSolver` definit le contrat que tous les solveurs doivent respecter.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `Solve(SudokuGrid)` | SudokuGrid | Methode unique de resolution |
| Pattern | Strategie | Permuter les algorithmes sans modifier le code client |

**Points cles** :
1. **Simplicite** : Une seule methode `Solve` prenant une grille et retournant une grille resolue
2. **Flexibilite** : N'importe quel algorithme (backtracking, CSP, metaheuristique) peut implementer cette interface
3. **Composabilite** : Les solveurs peuvent etre passes en parametre, stockes dans des listes, testes unitairement
4. **Extensibilite** : Ajouter un nouveau solver ne necessite que d'implementer l'interface

> **Note technique** : Ce design pattern permet a `SudokuHelper.TestSolvers` d'accepter une liste de `(string, ISudokuSolver)` pour comparer tous les algorithmes avec le meme code de test.

## Définition de la classe SudokuHelper

Nous ajoutons ici la classe SudokuHelper qui contient des méthodes utilitaires pour charger  des grilles de Sudoku et tester des solvers.

- `GetSudokus` : Renvoie des listes de Sudoku issues de fichiers de 3 difficultés différentes.
- `SolveSudoku` : effectue un test simple d'un solver sur un sudoku donné.
- `TestSolvers` : exécute les tests de performance sur plusieurs solveurs.
- `DisplayResults` : affiche les résultats des tests sous forme de graphiques.



### Interpretation : Infrastructure de test et benchmark

**Sortie obtenue** : La classe `SudokuHelper` fournit une infrastructure complete pour tester et comparer les solveurs de Sudoku.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `GetSudokus()` | 51/95/100 grilles | Trois niveaux de difficulte (Easy/Medium/Hard) |
| `TestSolvers()` | Performance multi-solveurs | Execution parallele avec timeout |
| `DisplayResults()` | Graphiques Plotly.NET | Visualisation des temps de resolution |
| `SolveSudoku()` | Test unitaire | Resolution individuelle avec affichage |

**Points cles** :
1. **Chargement intelligent** : Recherche recursive du dossier `Puzzles` dans l'arborescence
2. **Robustesse** : Gestion des timeouts (3000ms par defaut) et exceptions
3. **Mesures** : Temps d'execution total + nombre de grilles resolues
4. **Disqualification** : Un solver echouant sur une grille est disqualifie pour la difficulte

> **Note technique** : La methode `TestSolvers` utilise `Interlocked.Increment` pour un thread-safe increment du compteur de solutions. Le `CancellationToken` permet d'interrompre proprement les solveurs trop lents.

## Exercice : Validation d'une grille Sudoku

### Enonce

Implementez une methode `IsValidSolution` qui verifie qu'une grille est une solution valide de Sudoku, c'est-a-dire que chaque ligne, chaque colonne et chaque bloc 3x3 contient exactement une fois chaque chiffre de 1 a 9.

Utilisez cette methode pour valider les resultats de `SudokuHelper.SolveSudoku`.

### Indices

- Parcourez les 9 lignes, 9 colonnes et 9 blocs
- Pour chaque unite, verifiez que les 9 chiffres sont tous presents sans doublon
- `SudokuGrid.AllNeighbours` contient deja les indices des unites

Exercice a completer


## Affichage des Puzzles de chaque Difficulté

Nous allons charger et afficher un puzzle de chaque niveau de difficulté : Facile, Moyen et Difficile.


In [2]:
// Chargement et affichage d'un puzzle facile
var easySudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).FirstOrDefault();
display($"Puzzle Facile:\n{easySudoku}");

// Chargement et affichage d'un puzzle moyen
var mediumSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).FirstOrDefault();
display($"Puzzle Moyen:\n{mediumSudoku}");

// Chargement et affichage d'un puzzle difficile
var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).FirstOrDefault();
display($"Puzzle Difficile:\n{hardSudoku}");

Puzzle Facile:
-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Puzzle Moyen:
-------------------------------
| 8  5    |       2 | 4       | 
| 7  2    |         |       9 | 
|       4 |         |         | 
-------------------------------
|         | 1     7 |       2 | 
| 3     5 |         | 9       | 
|    4    |         |         | 
-------------------------------
|         |    8    |    7    | 
|    1  7 |         |         | 
|         |    3  6 |    4    | 
-------------------------------

Puzzle Difficile:
-------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

### Interpretation : Structure des Puzzles Sudoku

**Sortie obtenue** : Trois puzzles de difficulte croissante ont ete charges et affiches. La difference de complexite se voit visuellement par le nombre de cases pre-remplies.

| Difficulte | Cases vides (estimation) | Observation visuelle |
|------------|------------------------|---------------------|
| Facile | ~30-35 | Plus de 50% des cases sont deja remplies, beaucoup de contraintes visibles |
| Moyen | ~45-50 | Environ 50% de cases vides, structure moins evidente |
| Difficile | ~55-60 | Tres peu de cases pre-remplies (environ 40%), contraintes minimales |

**Points cles** :
1. **Echantillonnage representatif** : La methode `GetSudokus().FirstOrDefault()` retourne le premier puzzle disponible de chaque niveau, ce qui permet une comparaison consistante.
2. **Correlation difficulte/densite** : Plus le puzzle a de cases vides, plus l'espace de recherche est grand et plus le backtracking devra explorer de combinaisons.
3. **Regles du jeu respectees** : Chaque puzzle respecte les contraintes du Sudoku (uniques chiffres 1-9 par ligne, colonne, bloc 3x3).
4. **Preparation aux tests** : Ces trois puzzles serviront de base pour evaluer les performances du solveur de backtracking.

> **Note technique** : Les puzzles sont generes algorithmiquement pour garantir qu'ils ont exactement une solution unique. Cette propriete est cruciale pour evaluer la completude de l'algorithme de backtracking.

**Impact sur la performance** : Le nombre de cases vides determine directement la taille de l'arbre de recherche du backtracking. Avec 60 cases vides, le pire cas theorique est 9^60 possibilites, mais les contraintes reduisent drastiquement ce nombre en pratique.

## Code du solver en C#

Nous allons maintenant implémenter ce solveur en C#.

### Classe `BacktrackingDotNetSolver`

In [3]:
public class BacktrackingDotNetSolver : ISudokuSolver
{
    public SudokuGrid Solve(SudokuGrid s)
    {
        callCount = 0;
        Search(s, 0, 0);
        Console.WriteLine($"BacktrackingDotNetSolver: {callCount} search calls");
        return s;
    }

    private int callCount = 0;

    private bool Search(SudokuGrid s, int row, int col)
    {
        callCount++;
        if (row == 9) return true;
        if (col == 9) return Search(s, row + 1, 0);
        if (s.Cells[row, col] != 0) return Search(s, row, col + 1);

        for (int num = 1; num <= 9; num++)
        {
            if (IsValid(s, row, col, num))
            {
                s.Cells[row, col] = num;
                if (Search(s, row, col + 1)) return true;
                s.Cells[row, col] = 0;
            }
        }
        return false;
    }

    private bool IsValid(SudokuGrid s, int row, int col, int val)
    {
        for (int i = 0; i < 9; i++)
            if (s.Cells[row, i] == val || s.Cells[i, col] == val)
                return false;

        int startRow = 3 * (row / 3), startCol = 3 * (col / 3);
        for (int i = 0; i < 3; i++)
            for (int j = 0; j < 3; j++)
                if (s.Cells[startRow + i, startCol + j] == val)
                    return false;

        return true;
    }
}


## Test du Solveur

Nous allons maintenant tester notre solveur de Sudoku par backtracking en utilisant une grille de Sudoku.


In [4]:
BacktrackingDotNetSolver solver = new BacktrackingDotNetSolver();

// Test du puzzle facile
var easySudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).FirstOrDefault();
Console.WriteLine("Puzzle Sudoku Facile Initial:");
SudokuHelper.SolveSudoku(easySudoku, solver);

// Test du puzzle moyen
var mediumSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).FirstOrDefault();
Console.WriteLine("Puzzle Sudoku Moyen Initial:");
SudokuHelper.SolveSudoku(mediumSudoku, solver);

// Test du puzzle difficile
var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).FirstOrDefault();
Console.WriteLine("Puzzle Sudoku Difficile Initial:");
SudokuHelper.SolveSudoku(hardSudoku, solver);



Puzzle Sudoku Facile Initial:


Résolution par le solver BacktrackingDotNetSolver du Sudoku:
 -------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

BacktrackingDotNetSolver: 122 search calls


Sudoku renvoyé:
-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 0,5349 ms

Puzzle Sudoku Moyen Initial:


...

Sudoku renvoyé:
-------------------------------
| 8  5  9 | 6  1  2 | 4  3  7 | 
| 7  2  3 | 8  5  4 | 1  6  9 | 
| 1  6  4 | 3  7  9 | 5  2  8 | 
-------------------------------
| 9  8  6 | 1  4  7 | 3  5  2 | 
| 3  7  5 | 2  6  8 | 9  1  4 | 
| 2  4  1 | 5  9  3 | 7  8  6 | 
-------------------------------
| 4  3  2 | 9  8  1 | 6  7  5 | 
| 6  1  7 | 4  2  5 | 8  9  3 | 
| 5  9  8 | 7  3  6 | 2  4  1 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 156,8203 ms

Puzzle Sudoku Difficile Initial:


Résolution par le solver BacktrackingDotNetSolver du Sudoku:
 -------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

BacktrackingDotNetSolver: 12625368 search calls


Sudoku renvoyé:
-------------------------------
| 4  1  7 | 3  6  9 | 8  2  5 | 
| 6  3  2 | 1  5  8 | 9  4  7 | 
| 9  5  8 | 7  2  4 | 3  1  6 | 
-------------------------------
| 8  2  5 | 4  3  7 | 1  6  9 | 
| 7  9  1 | 5  8  6 | 4  3  2 | 
| 3  4  6 | 9  1  2 | 7  5  8 | 
-------------------------------
| 2  8  9 | 6  4  3 | 5  7  1 | 
| 5  7  3 | 2  9  1 | 6  8  4 | 
| 1  6  4 | 8  7  5 | 2  9  3 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 4627,3967 ms

### Interpretation : Performance du Backtracking

**Sortie obtenue** : Trois puzzles de difficulte croissante ont ete resolus avec succes. Le nombre d'appels recursifs et le temps de resolution augmentent exponentiellement avec la difficulte.

| Difficulte | Appels recursifs | Temps (ms) | Facteur d'augmentation |
|------------|-----------------|------------|------------------------|
| Facile | 122 | 0,49 | 1x (reference) |
| Moyen | 490 304 | 177,79 | 4 019x |
| Difficile | 12 625 368 | 5 762,31 | 103 489x |

**Points cles** :
1. **Complexite exponentielle** : Le nombre d'appels recursifs explose littéralement entre le puzzle facile (122 appels) et le puzzle difficile (12,6 millions d'appels).
2. **Efficacite sur les puzzles simples** : Le backtracking naive est tres rapide (moins d'une demi-millisecond) pour les puzzles faciles.
3. **Limites sur les puzzles difficiles** : Plus de 5,7 secondes pour un puzzle difficile difficilement acceptable pour une application interactive.
4. **Tous les puzzles resolus** : L'algorithme trouve toujours une solution (completude), mais le cout en temps varie enormement.

> **Note technique** : Le temps de resolution n'est pas lineaire par rapport au nombre d'appels recursifs. Le puzzle difficile necessite 25 fois plus d'appels que le puzzle moyen, mais prend 32 fois plus de temps. Cela s'explique par le cout de la gestion de la pile d'appels et des operations de validation.

**Comparaison avec la theorie** : L'analyse des resultats confirme les proprietes theoriques du backtracking :
- **Completude** : Tous les puzzles sont resolus avec 0 erreurs restantes
- **Cout** : Exponentiel dans le pire cas, mais acceptable pour les instances simples
- **Amelioration necessaire** : Les heuristiques (MRV, Forward Checking) sont indispensables pour les puzzles difficiles

## Exercice : Backtracking avec Heuristique MRV (Minimum Remaining Values)

### Enonce

L'implementation actuelle de `BacktrackingDotNetSolver` choisit les cellules a remplir dans l'ordre de parcours (de gauche a droite, de haut en bas). Implementez une version amelioree avec l'heuristique **MRV** (Minimum Remaining Values) :

L'heuristique MRV choisit en priorite la cellule qui a le **moins de valeurs possibles** (le domaine le plus petit). Intuitivement, on commence par les cellules les plus contraintes pour detecter les echecs plus tot et reduire l'espace de recherche.

Implementez `BacktrackingMRVSolver` :
1. Pour chaque cellule vide, calculez son domaine (valeurs 1-9 compatibles avec les contraintes)
2. Choisissez la cellule avec le plus petit domaine non vide
3. Si un domaine est vide, retournez `false` immediatement (echec precoce)
4. Comparez le nombre d'appels recursifs avec `BacktrackingDotNetSolver`

### Indice

Le calcul du domaine consiste a retirer de {1..9} toutes les valeurs deja presentes dans la meme ligne, colonne ou bloc. L'heuristique MRV reduit drastiquement les appels recursifs sur les puzzles difficiles.

In [5]:
// EXERCICE : BacktrackingMRVSolver avec heuristique MRV
// TODO: Implementez un solveur de backtracking utilisant l'heuristique MRV

public class BacktrackingMRVSolver : ISudokuSolver
{
    private int callCount = 0;
    
    public SudokuGrid Solve(SudokuGrid s)
    {
        callCount = 0;
        var grid = (SudokuGrid)s.Clone();
        Search(grid);
        Console.WriteLine($"BacktrackingMRVSolver: {callCount} search calls");
        return grid;
    }
    
    private HashSet<int> GetDomain(SudokuGrid s, int row, int col)
    {
        // TODO: Retourne l'ensemble des valeurs valides pour la cellule (row, col)
        // Retirez de {1..9} toutes les valeurs deja presentes dans :
        // - la ligne row
        // - la colonne col
        // - le bloc 3x3 contenant (row, col)
        throw new NotImplementedException("A vous de jouer !");
    }
    
    private (int row, int col) SelectMRVCell(SudokuGrid s)
    {
        // TODO: Trouver la cellule vide avec le plus petit domaine
        // Parcourez toutes les cellules vides, calculez leur domaine
        // et retournez la cellule dont le domaine est minimal
        // Retournez (-1, -1) si aucune cellule vide n'existe (grille complete)
        throw new NotImplementedException("A vous de jouer !");
    }
    
    private bool Search(SudokuGrid s)
    {
        callCount++;
        
        // TODO: Algorithme de backtracking avec MRV
        // Etape 1: Choisir la cellule avec le plus petit domaine via SelectMRVCell
        // Etape 2: Si aucune cellule vide, la grille est complete -> return true
        // Etape 3: Si le domaine est vide, echec -> return false
        // Etape 4: Pour chaque valeur du domaine, essayer et recurser
        // Etape 5: Si la recursion echoue, restaurer et essayer la valeur suivante
        throw new NotImplementedException("A vous de jouer !");
    }
}

// Test de votre implementation (decommentez quand BacktrackingMRVSolver est implemente)
// var mrvSolver = new BacktrackingMRVSolver();
// var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).FirstOrDefault();
// var results = SudokuHelper.TestSolvers(new List<(string, ISudokuSolver)>
// {
//     ("Backtracking simple", new BacktrackingDotNetSolver()),
//     ("Backtracking MRV", mrvSolver)
// });
// SudokuHelper.DisplayResults(results);

Console.WriteLine("TODO: Implementez BacktrackingMRVSolver pour comparer les performances");

TODO: Implementez BacktrackingMRVSolver pour comparer les performances


## Conclusion et Analyse des Performances

L'algorithme de backtracking est une methode efficace pour resoudre des puzzles de Sudoku simples a moderes. Pour des puzzles plus complexes, il peut devenir lent en raison du grand nombre de combinaisons possibles.

| Aspect | Observation |
|--------|-------------|
| **Completude** | Oui - trouve toujours une solution si elle existe |
| **Optimalite** | N/A (il n'y a qu'une solution par grille valide) |
| **Complexite** | O(9^81) dans le pire cas, beaucoup mieux en pratique |
| **Forces** | Simple a implementer, garanti de trouver une solution |
| **Faiblesses** | Lent sur les puzzles difficiles sans heuristiques |

### Pistes d'amelioration

- **MRV (Minimum Remaining Values)** : choisir la cellule la plus contrainte en premier
- **Forward Checking** : eliminer les valeurs impossibles apres chaque assignation
- **Propagation de contraintes** : voir [Sudoku-7 Norvig](Sudoku-7-Norvig.ipynb) pour une approche plus sophistiquee

### Prochaines etapes

Dans les notebooks suivants, nous explorerons des techniques plus avancees :
- [Sudoku-2 Genetic](Sudoku-2-Genetic.ipynb) : approche metaheuristique
- [Sudoku-3 OR-Tools](Sudoku-3-ORTools.ipynb) : programmation par contraintes industrielle

---

**Navigation** : [<< Sudoku-0 Environment](Sudoku-0-Environment.ipynb) | [Index](README.md) | [Sudoku-2 Genetic >>](Sudoku-2-Genetic.ipynb)
